# Suite DOM — Value domains

A value domain names a standard's representation (`Integer`, `Ieee754`, `Bits`, `Bytes`, `Unicode`, `Ieee1164`, `Enum`,
`Packed`, `Bool`), with only its parameters. Domains are data: a dialect, a singleton, owns each by name, and each one's
schema is a named `Schemas.OfAny` of the scalar it narrows, related to it both ways; they are stored, validated and
round-trip. A term carries no domain (the port's stage 3, Decided, 1), so evaluation takes values by the schemas they are
written with, ints unbounded and floats binary64; what a domain's values mean, its kind's `contains`, `compare` and
`fit`, and IEEE 754's arithmetic, are kept for domain inference, a later capability (Decided, 2), and tested here
directly.

In [ ]:
import { Evaluators as V, Expressions as E, Framework } from "@mbse/expressions";
import * as IEEE1164 from "@mbse/expressions/IEEE1164/Expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as CDialect from "@mbse/expressions/C/Expressions";
import * as Cpp from "@mbse/expressions/Cpp/Expressions";
import * as Excel from "@mbse/expressions/Excel/Expressions";
import * as Latex from "@mbse/expressions/LaTeX/Expressions";
import * as Matlab from "@mbse/expressions/MATLAB/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as Patterns from "@mbse/expressions/Framework/Patterns";
import * as Translators from "@mbse/expressions/Translators";
import { Domains as FD, Evaluators as FE, Terms as F } from "@mbse/expressions/Framework";
import { OverflowError } from "@mbse/expressions/Framework/Errors";
import { Builders, Comparison as C, Errors, JSON, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

const { Domains: D, Ieee754 } = Patterns;
const { LookupError, NotImplementedError, ValueError } = Errors;

const INSTALLS = [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install, Patterns.install];
const store: any = new Builders.Store();
for (const install of INSTALLS) store.transaction(install); // the framework, the standard dialects, then Patterns: each its own transaction
const Term: any = store.registered("Expressions.Framework.Term");

/** The term value `spec` writes, built in a transaction. */
function term(spec: unknown): any {
  return store.transaction(() => store.Expressions.Framework.Term(E.DIALECT.spec(spec)));
}

/** The value of the term `spec` writes, with the variables in `scope` bound. */
function val(spec: unknown, scope: Record<string, unknown> = {}): any {
  return V.OfAny(term(spec), scope);
}

/** The value of a Patterns call, with the variables in `scope` bound. */
function evIn(scope: Record<string, unknown>, name: string, ...args: unknown[]): any {
  return val(E.operation(name, ...args), scope);
}

/** The value of a Patterns call. */
function ev(name: string, ...args: unknown[]): any {
  return evIn({}, name, ...args);
}

const bytes = (...values: number[]) => new Uint8Array(values);
const I8 = new D.OfInteger.Builder().width(8n).create();
const U8 = new D.OfInteger.Builder().width(8n).signed(false).overflow("wrap").create();
const F32 = new D.OfIeee754.Builder().format("binary32").create();
const [F16, F128] = [new D.OfIeee754.Data("binary16"), new D.OfIeee754.Data("binary128")];
const [D64, D128] = [new D.OfIeee754.Data("decimal64"), new D.OfIeee754.Data("decimal128")];
const Bits2 = new D.OfBits.Builder().width(2n).create();
const State = new D.OfEnum.Builder().members("IDLE", "RUN").create();
const Coded = new D.OfPacked.Builder().domain(State).representation(Bits2).codes(0n, 1n).create();
const Nib = new D.OfBits.Data(4n);
const U = new D.OfInteger.Data(null, false);
const x = E.variable("x");

<a id="DOM-01"></a>
## DOM-01 · Value domains name standards, with only their parameters, are compared by structure and validated

In [ ]:
{
  const names: [Patterns.Domains.Domain, string][] = [[D.Bool, "bool"], [D.Int, "int"], [I8, "int8"], [U8, "uint8"],
    [new D.OfInteger.Builder().signed(false).create(), "uint"], [D.Float, "float"], [F32, "binary32"],
    [new D.OfIeee754.Builder().rounding("roundTowardZero").create(), "binary64 roundTowardZero"],
    [Bits2, "bits2"], [D.Bytes, "bytes"], [new D.OfBytes.Builder().width(4n).create(), "bytes4"], [D.Str, "str"],
    [new D.OfIeee1164.Builder().create(), "std_logic"], [new D.OfIeee1164.Builder().width(4n).create(), "std_logic_vector4"], [State, "enum(IDLE, RUN)"], [Coded, "packed(enum(IDLE, RUN) as bits2)"]];
  assert(names.every(([domain, name]) => domain.name() === name && String(domain) === name));
  assert(names.every(([domain]) => domain.validate().length === 0));
  // The natives' defaults, compared by structure: an equal domain built anew is the same domain.
  assert(new D.OfInteger.Data().equals(D.Int) && new D.OfIeee754.Data("binary64").equals(D.Float) && new D.OfUnicode.Data().equals(D.Str));
  assert(new D.OfBytes.Data().equals(D.Bytes) && new D.OfBool.Data().equals(D.Bool) && !I8.equals(U8) && !I8.equals(D.Int));
  assert(new D.OfInteger.Builder().width(8n).create().equals(I8) && String(D.OfInteger) === "<domain kind integer>");
  assert(D.FORMATS[2] === "binary64" && D.ROUNDINGS[0] === "roundTiesToEven" && same(D.OVERFLOWS, ["wrap", "saturate", "raise"]));
  assert(D.STATES.length === 9);
  // Each kind's problems.
  const problems: [Patterns.Domains.Domain, string[]][] = [
    [new D.OfInteger.Data(0n), ["a width must be a positive int, got 0"]],
    [new D.OfInteger.Data(null, false, "wrap"), ["an integer without a width cannot wrap"]],
    [new D.OfInteger.Data(8n, "yes" as never, "clip"), ["signed must be a bool, got 'yes'",
      "overflow must be one of wrap, saturate, raise, got 'clip'"]],
    [new D.OfIeee754.Data("binary8", "up"), [
      "format must be one of binary16, binary32, binary64, binary128, decimal64, decimal128, got 'binary8'",
      "rounding must be one of roundTiesToEven, roundTiesToAway, roundTowardPositive, roundTowardNegative, " +
      "roundTowardZero, got 'up'"]],
    [new D.OfBits.Data(), ["a bits domain needs a width"]],
    [new D.OfBits.Data(0n), ["a width must be a positive int, got 0"]],
    [new D.OfBytes.Data(-1n), ["a width must be a positive int, got -1"]],
    [new D.OfEnum.Data(), ["an enum needs a member"]],
    [new D.OfEnum.Data(["A", "", "A"]), ["member 1 has no name", "member 'A' appears twice"]],
    [new D.OfPacked.Data(I8, Bits2, []), ["a packed domain packs an enum, got int8"]],
    [new D.OfPacked.Data(), ["a packed domain packs an enum, got None"]],
    [new D.OfPacked.Data(State, D.Bytes, [0n, 1n]), ["a packed domain's representation is an integer or bits of a width, got bytes"]],
    [new D.OfPacked.Data(State, D.Int, [0n, 1n]), ["a packed domain's representation is an integer or bits of a width, got int"]],
    [new D.OfPacked.Data(State, Bits2, [0n]), ["a packed domain needs a code per member: 2 members, 1 codes"]],
    [new D.OfPacked.Data(State, new D.OfBits.Data(1n), [0n, 2n]), ["code 1 does not fit bits1: 2"]],
    [new D.OfPacked.Data(State, Bits2, [1n, 1n]), ["code 1 is used twice"]],
    [new D.OfPacked.Data(State, new D.OfInteger.Data(2n), [-2n, 2n]), ["code 1 does not fit int2: 2"]],
    [new D.OfPacked.Data(State, new D.OfInteger.Data(2n), ["0" as never, 1n]), ["code 0 does not fit int2: '0'"]],
    [new D.OfPacked.Data(new D.OfEnum.Data(), new D.OfInteger.Data(2n, true, "x"), []), [
      "domain: an enum needs a member", "representation: overflow must be one of wrap, saturate, raise, got 'x'"]],
  ];
  for (const [domain, expected] of problems) assert(same(domain.validate(), expected), `${domain}: ${domain.validate()}`);
  assert(new D.OfPacked.Data().name() === "packed(None as None)");
  // Builders finalize as every builder does, and set each parameter.
  const built = new D.OfIeee754.Builder(F32).rounding("roundTowardNegative").clone();
  assert(built.equals(new D.OfIeee754.Data("binary32", "roundTowardNegative")) && F32.rounding === "roundTiesToEven");
  assert(new D.OfBytes.Builder(new D.OfBytes.Data()).width(3n).update().width === 3n);
  assert(same(new D.OfIeee1164.Data(0n).validate(), ["a width must be a positive int, got 0"]));
  for (const builder of [new D.OfBool.Builder(), new D.OfUnicode.Builder(), new D.OfIeee1164.Builder()]) {
    assert(builder.create().validate().length === 0);
  }
  raises(ValueError, () => new D.OfInteger.Builder(I8).create(), "create() is only valid without a source instance; use clone() or update()");
  raises(ValueError, () => new D.OfInteger.Builder().clone(), "clone() is only valid with a source instance");
  raises(ValueError, () => new D.OfInteger.Builder().update(), "update() is only valid with a source instance");
  raises(TypeError, () => new D.OfInteger.Builder(Bits2 as never), "expected integer data to build from, got bits2");
}

<a id="DOM-02"></a>
## DOM-02 · A domain holds the values of its native type that fit it, and includes only itself

In [ ]:
{
  const holds: [Patterns.Domains.Domain, unknown[], unknown[]][] = [
    [I8, [127n, -128n], [128n, -129n, true, 1.0]], [U8, [0n, 255n], [-1n, 256n]], [D.Int, [10n ** 30n, -5n], [false]],
    [new D.OfInteger.Data(null, false), [10n ** 30n], [-1n]], [new D.OfInteger.Data(0n), [10n ** 30n], []], // not a width: unbounded
    [F32, [1.5, NaN], [1n, "1.5"]], [new D.OfIeee754.Data("decimal64"), ["0.1"], [0.1]],
    [new D.OfBits.Data(3n), [bytes(7), bytes(0)], [bytes(8), bytes(0, 7), "7"]], [new D.OfBits.Data(8n), [bytes(255)], [bytes()]],
    [new D.OfBits.Data(9n), [bytes(1, 255)], [bytes(2, 0), bytes(255)]], [new D.OfBits.Data(), [], [bytes(0)]],
    [new D.OfBytes.Data(2n), [bytes(97, 98)], [bytes(97)]], [D.Bytes, [bytes(), bytes(97, 98, 99)], ["abc"]],
    [D.Str, ["", "日本語 🚀"], ["\ud800", bytes(97)]], [new D.OfIeee1164.Data(), ["Z", "-", "U"], ["z", "2", 1n]],
    [new D.OfIeee1164.Data(4n), ["10XZ", "UUUU"], ["10x", "10XZ1", "10xz", 1n]],
    [State, ["RUN"], ["STOP", 0n]], [Coded, ["IDLE"], ["STOP"]], [new D.OfPacked.Data(), [], ["IDLE"]],
    [D.Bool, [true, false], [1n, 0n]],
  ];
  for (const [domain, inside, outside] of holds) {
    assert(inside.every((v) => domain.contains(v)) && !outside.some((v) => domain.contains(v)), String(domain));
  }
  assert(new D.OfIeee754.Data("decimal128").native() === String && F32.native() === Number && I8.native() === BigInt);
  // A domain includes only domains equal to it: Patterns has no implied promotions.
  assert(I8.includes(new D.OfInteger.Data(8n)) && !I8.includes(D.Int) && !D.Int.includes(I8) && !I8.includes(U8));
  assert(I8.includes(new FD.OfUnion(I8, new D.OfInteger.Data(8n))) && !I8.includes(new FD.OfUnion(I8, U8)));
  assert(FD.overlaps(FD.Anything, I8) && !D.Float.includes(F32));
  // A native's default domain, and the domains of other values.
  assert(same([true, 1n, 1.5, "a", bytes(97)].map((v) => D.of(v)), [D.Bool, D.Int, D.Float, D.Str, D.Bytes]));
  assert(D.of(term(E.variable("x"))) === D.Object); // a term value is an object
  raises(TypeError, () => D.of(null), "a NoneType is not a value of the Patterns dialect");
}

<a id="DOM-03"></a>
## DOM-03 · Domains are data: a dialect owns each by name, and each one's schema, an OfAny of its scalar, reads it

In [ ]:
const DIALECTS = new Map<any, string>([[ISOIEC11404, "ISOIEC11404"], [IEEE754, "IEEE754"], [IEEE1164, "IEEE1164"], [Unicode, "Unicode"]]); // each by its family
const STANDARD = new Map<string, Map<string, string>>([ // the standard dialects' domains, each the scalar its schema narrows
  ["ISOIEC11404", new Map([["boolean", "bool"], ["integer", "int"], ["octetstring", "bytes"], ["uint8", "int"]])],
  ["IEEE754", new Map([["binary64", "float"], ["binary32", "float"]])], ["IEEE1164", new Map([["std_logic", "str"]])],
  ["Unicode", new Map([["string", "str"]])]]);
const defined = new Map<string, any>([...DIALECTS].map(([module, family]) => [family, store.transaction(module.define)[0]])); // each dialect's domains
const singletons = new Map<string, any>([...STANDARD.keys()].map((family) => [family, store.Instances.Expressions[family].Expressions])); // by its path
assert([...STANDARD.keys()].every((family) => singletons.get(family) === defined.get(family).get("dialect")));
const iso = singletons.get("ISOIEC11404");
const domains = new Map<string, any>([...STANDARD.keys()].flatMap((family) =>
  [...singletons.get(family).domains].map((e: any): [string, any] => [`${e.name}`, e.domain])));
assert(equal(new Map([...STANDARD.keys()].map((family) => [family, [...singletons.get(family).domains].map((e: any) => `${e.name}`)])),
  new Map([...STANDARD].map(([family, names]) => [family, [...names.keys()]]))));
assert(iso.domains({ name: "uint8" }) === domains.get("uint8") && domains.get("uint8") === defined.get("ISOIEC11404").get("uint8"));
assert(store.transaction(ISOIEC11404.define)[0].get("uint8") === domains.get("uint8")); // again: nothing
const expectedDomains: [string, Patterns.Domains.Domain][] = [["boolean", D.Bool], ["integer", D.Int], ["octetstring", D.Bytes],
  ["uint8", U8], ["binary64", D.Float], ["binary32", F32], ["std_logic", new D.OfIeee1164.Data()], ["string", D.Str]]; // the domains Patterns held, renamed by their standards
assert(domains.size === expectedDomains.length && expectedDomains.every(([name, domain]) => D.read(domains.get(name)).equals(domain)));
const scalarSchemas: Record<string, string> = { bool: "Schemas.OfBool", int: "Schemas.OfInt", float: "Schemas.OfFloat", str: "Schemas.OfStr", bytes: "Schemas.OfBytes" };
for (const [family, names] of STANDARD) { // each one's schema: an OfAny of its scalar, its domain read both ways
  for (const [name, scalar] of names) {
    const schema = store.registered(`Expressions.${family}.Expressions.${name}`);
    assert(same(schema.schemas, [store.registered(scalarSchemas[scalar] as string)]) && schema.validate().length === 0);
    assert(same([...domains.get(name).schema].map((e: any) => e.schema), [schema]));
    const read = [...new Builders.OfAny.Builder(schema, false, store).domain].map((e: any) => D.read(e.domain));
    assert(read.length === 1 && read[0]!.equals(D.read(domains.get(name))));
  }
}
const [Domain, HasDomain]: any[] = [store.registered("Expressions.Framework.Domain"), store.registered("Expressions.Framework.HasDomain")];
assert(equal(HasDomain.uniques.map((u: any) => [...u]), [["schema"], ["domain"]]) && Domain.adjacencies.get("dialect").owner);
// The framework's union of kinds is of the dialects' unions, each named by its dialect's tag.
const DomainKind = store.registered("Expressions.Framework.DomainKind");
assert(same(DomainKind.branches.map((b: any) => b.name), ["isoiec11404", "ieee754", "ieee1164", "unicode"]));
assert(same(DomainKind.branches[0].schema.branches.map((b: any) => b.name), ["bool", "integer", "bits", "bytes", "enum", "packed"]));
// A domain of its own, owned by its dialect by name; a packed one links its enum and its representation, domains too.
const [state, coded, bad, empty] = store.transaction(() => {
  const state = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union(
    (u: any) => u.isoiec11404((v: any) => v.enum((e: any) => e.members(["IDLE", "RUN"]).ordered(false)))))));
  const coded = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union((u: any) => u.isoiec11404(
    (v: any) => v.packed((e: any) => e.codes([0n, 1n]).ordered(true)))))).packs((a: any) => a.insert(
    (e: any) => e.domain(state).representation(domains.get("uint8")))));
  const bad = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union(
    (u: any) => u.isoiec11404((v: any) => v.integer((e: any) => e.width(0n)))))));
  const empty = store.Expressions.Framework.Domain((d: any) => d);
  store.Expressions.ISOIEC11404.Expressions.Dialect(iso, (b: any) => b.domains({
    state: (e: any) => e.domain(state), coded: (e: any) => e.domain(coded), bad: (e: any) => e.domain(bad),
    empty: (e: any) => e.domain(empty) })); // each owned: written with its owner
  return [state, coded, bad, empty];
});
const OWN = [...(STANDARD.get("ISOIEC11404") as Map<string, string>).keys(), "state", "coded", "bad", "empty"];
assert(same([...iso.domains].map((e: any) => `${e.name}`), OWN));
assert(D.read(state).equals(State) && D.read(coded).equals(new D.OfPacked.Data(State, U8, [0n, 1n])) && D.read(coded).validate().length === 0);
assert(D.read(coded).name() === "packed(enum(IDLE, RUN) as uint8)" && D.read(coded).contains("RUN"));
assert(same(D.read(bad).validate(), ["a width must be a positive int, got 0"])); // validated as its kind's data
raises(ValueError, () => D.read(empty), "a domain needs a kind");

<a id="DOM-05"></a>
## DOM-05 · Over the wire a domain is its dialect's, by name: it round-trips, and so does a store holding it

In [ ]:
{
  const Dialect = store.registered("Expressions.ISOIEC11404.Expressions.Dialect");
  // Each a document of its own, by its path: read back, it is the object the store holds. Its kind is written its
  // dialect's, then its kind's.
  assert(JSON.ToJSON(store)(Domain, domains.get("uint8")).includes('"isoiec11404": {"integer": {"width": 8, "signed": false, "overflow": "wrap", "least": 0, "greatest": 255, '
    + '"ordered": true}}'));
  assert(JSON.ToJSON(store)(Domain, domains.get("binary32")).includes('"ieee754": {"ieee754": {"format": "binary32", "rounding": "roundTiesToEven",'));
  for (const domain of [domains.get("uint8"), domains.get("binary32"), state, coded]) {
    for (const [encode, decode] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]) {
      const back = decode(Domain, encode(Domain, domain));
      assert(back === domain && D.read(back).equals(D.read(domain))); // owned: its path, its dialect's and its name, finds it
    }
  }
  const back: any = JSON.FromJSON(store)(Dialect, JSON.ToJSON(store)(Dialect, iso));
  assert(back === iso && same([...back.domains].map((e: any) => `${e.name}`), OWN));
  // A store holding them round-trips whole, after its documents were read back, into a store with the same installs: nothing lost, nothing written twice.
  const snapshot = JSON.ToJSON(store).Snapshot();
  const again: any = new Builders.Store();
  for (const install of INSTALLS) again.transaction(install);
  JSON.FromJSON(again).Snapshot(snapshot);
  assert(JSON.ToJSON(again).Snapshot() === snapshot);
  const restored = new Map<string, any>([...again.Instances.Expressions.ISOIEC11404.Expressions.domains].map((e: any) => [`${e.name}`, e.domain])); // keyed by name
  assert(same([...restored.keys()].sort(), [...OWN].sort()));
  assert(same(["state", "coded", "bad"].map((name) => D.read(restored.get(name)).name()), [
    "enum(IDLE, RUN)", "packed(enum(IDLE, RUN) as uint8)", "int0"]));
  // A term carries no domain: a literal holds a value, its spelling and its problems, and its value's schema is its own.
  assert(same([...store.registered("Expressions.Framework.Literal").properties.keys()], ["value", "text", "problems"])
    && E.literal.length === 1);
}

<a id="DOM-06"></a>
## DOM-06 · A domain compares two of its values, ordered by kind; evaluation compares values by their schemas

In [ ]:
{
  assert(I8.compare(-1n, 1n) === -1 && I8.compare(5n, 5n) === 0 && I8.kind().ORDERED && F32.kind().ORDERED && D.Bytes.kind().ORDERED);
  assert(!State.kind().ORDERED && !Bits2.kind().ORDERED && !new D.OfIeee1164.Data().kind().ORDERED && Coded.kind().ORDERED);
  assert(new D.OfBytes.Data(2n).compare(bytes(0, 1), bytes(1, 0)) === -1 && State.compare("RUN", "RUN") === 0);
  assert(Coded.compare("RUN", "RUN") === 0 && Coded.compare("IDLE", "RUN") === -1 && new D.OfPacked.Data(State, Bits2, [1n, 0n]).compare("IDLE", "RUN") === 1); // by code
  // A format's floats compare as mbse-schemas compares floats: NaNs equal each other, and -0 comes before 0.
  assert(F32.compare(NaN, NaN) === 0 && F32.compare(NaN, 1.0) === null && F32.compare(-0, 0) === -1);
  // Evaluation compares values by the schemas they are written with: of one scalar, else incomparable.
  assert(ev("lt", -1n, 1n) === true && ev("eq", 5n, 5.0) === null && ev("eq", "RUN", "RUN") === true);
}


<a id="DOM-07"></a>
## DOM-07 · An integer domain fits a result by its overflow; evaluation's ints are unbounded

In [ ]:
{
  assert(I8.fit("add", 127n) === 127n && equal(I8.bounds(), [-128n, 127n]) && equal(U.bounds(), [0n, null]) && equal(D.Int.bounds(), [null, null]));
  raises(OverflowError, () => I8.fit("add", 128n), "add overflows int8: 128");
  assert(U8.fit("add", 260n) === 4n && U8.fit("sub", -1n) === 255n); // wraps
  const [W8, S8] = [new D.OfInteger.Data(8n, true, "wrap"), new D.OfInteger.Data(8n, true, "saturate")];
  assert(W8.fit("mul", 128n) === -128n && S8.fit("add", 200n) === 127n && S8.fit("sub", -200n) === -128n);
  // An unbounded unsigned integer has a least value only, and without a width wrap does not apply.
  const US = new D.OfInteger.Data(null, false, "saturate");
  assert(US.fit("sub", -1n) === 0n && US.fit("add", 10n ** 30n) === 10n ** 30n);
  raises(OverflowError, () => U.fit("sub", -1n), "sub overflows uint: -1");
  raises(OverflowError, () => new D.OfInteger.Data(null, false, "wrap").fit("sub", -1n), "sub overflows uint: -1"); // invalid, as validation reports; fitting raises
  // Evaluation takes ints by their schema, unbounded, and floats as binary64; it does not promote.
  assert(ev("add", 2n ** 70n, 1n) === 2n ** 70n + 1n && ev("neg", -128n) === 128n && ev("mul", 1.5, 2.0) === 3.0);
  raises(TypeError, () => ev("add", 1n, 1.5), "add expects numbers of one domain, got int and float");
  raises(TypeError, () => ev("neg", "a"), "neg expects a number, got str");
}


<a id="DOM-08"></a>
## DOM-08 · The bitwise operations take ints, as two's complement of infinite width, and shifts an int count

In [ ]:
{
  assert(ev("bitand", 12n, 10n) === 8n && ev("bitor", 12n, 10n) === 14n && ev("bitxor", 12n, 10n) === 6n && ev("bitnot", 5n) === -6n);
  assert(ev("bitand", -1n, 15n) === 15n && ev("bitnot", -1n) === 0n && ev("bitor", -16n, 15n) === -1n);
  assert(ev("shl", 3n, 4n) === 48n && ev("shr", -7n, 1n) === -4n && ev("shl", 1n, 100n) === 2n ** 100n); // shr toward negative infinity
  raises(TypeError, () => ev("bitand", 1n, bytes(1)), "bitand expects integers of one domain, got int and bytes");
  raises(TypeError, () => ev("bitnot", true), "bitnot expects an integer, got bool");
  raises(ValueError, () => ev("shl", 1n, -1n), "shl needs a non-negative count, got -1");
  raises(TypeError, () => ev("shr", 1n, 1.0), "shr expects an integer count, got float");
  raises(TypeError, () => ev("shl", "a", 1n), "shl expects an integer, got str");
  assert(evIn({ x: null }, "bitand", x, 1n) === null && evIn({ x: null }, "shl", x, 1n) === null);
  assert(evIn({ x: null }, "shl", 1n, x) === null);
}


<a id="DOM-09"></a>
## DOM-09 · Signatures take any domain of a family, so inference follows the domains an environment gives

In [ ]:
function infer(spec: unknown, environment: Record<string, any> = {}): any {
  return E.DIALECT.infer(term(spec), environment);
}

{
  const y = E.variable("y");
  assert(infer(x.eq(y), { x: State, y: State }) === D.Bool && infer(x.lt(y), { x: I8, y: I8 }) === D.Bool);
  raises(TypeError, () => infer(x.lt(y), { x: State, y: State }),
    "lt cannot take (enum(IDLE, RUN), enum(IDLE, RUN)); it takes (T, T) -> bool for T in " +
    "int | float | str | bytes or any integer, ieee754, bytes, unicode or packed domain");
  assert(infer(x.mul(y), { x: F32, y: FD.Anything }).equals(F32)); // an unknown takes the other's domain
  assert(infer(x.bitxor(y), { x: Nib, y: Nib }).equals(Nib) && infer(E.operation("bitnot", 1n)) === D.Int);
  assert(infer(x.shl(y), { x: U8, y: I8 }).equals(U8) && infer(E.operation("shr", x, 1n), { x: FD.Anything }) === D.Int);
  assert(infer(E.operation("shl", 1n, x), { x: new FD.OfUnion(D.Int, I8) }) === D.Int);
  raises(TypeError, () => infer(E.operation("shl", 1n, 1.5)),
    "shl cannot take (int, float); it takes (T, any integer domain) -> T for T in int or any integer or bits domain");
  raises(TypeError, () => infer(E.operation("shr", 1.5, 1n)), "shr cannot take (float, int)");
  assert(E.DIALECT.validate(term(E.operation("bitand", 1n, 2n)), [], true).length === 0);
  // A family includes every domain of its kinds, and holds their values.
  const family = (D.SIGNATURES.get("add") as FD.Same).family as FD.Domain;
  assert(String(family) === "integer or ieee754 domain" && family.contains(1n) && family.contains(1.5));
  assert(!family.contains("a") && family.includes(I8) && !family.includes(State));
}


<a id="DOM-10"></a>
## DOM-10 · An IEEE 754 value is a float up to binary64 and canonical text otherwise, which the format must hold

In [ ]:
{
  const holds: [Patterns.Domains.Domain, unknown[], unknown[]][] = [
    [F16, [1.5, 65504.0, 2 ** -24, -0, Infinity, NaN], [0.1, 1e10, 2 ** -25, "1.5"]],
    [F32, [0.10000000149011612], [0.1]],
    [new D.OfIeee754.Data("binary64", "roundTowardZero"), [0.1, 5e-324], ["0.1"]],
    [F128, ["0.1", "1E+2", "-0", "Infinity", "-Infinity", "NaN", "1.189731495357231765085759326628007E+4932"],
      ["0.10", "100", "1e2", "-NaN", "1E+5000", "0.1 ", 0.1]],
    [D64, ["1.50", "1.000000000000000E+16", "0E-398", "-0", "0.000001", "1E-7", "NaN"],
      ["1.5E+0", "12345678901234567", "1E+385", "1E-399", "0.0000001", "+1", "abc"]],
    [D128, ["1.000000000000000000000000000000000", "2E+6111"], ["1E+6112"]],
  ];
  for (const [domain, inside, outside] of holds) {
    assert(inside.every((v) => domain.contains(v)) && !outside.some((v) => domain.contains(v)), String(domain));
  }
  assert(F128.native() === String && D64.native() === String && F16.native() === Number);
  assert(!new D.OfIeee754.Data("binary8").contains(1.5)); // an unknown format holds nothing
  // A decimal value's text is General Decimal Arithmetic's scientific form, keeping the coefficient and the exponent.
  const data: [boolean, bigint, number][] = [[false, 15n, -1], [true, 150n, -2], [false, 1n, 2], [false, 123n, -9], [false, 0n, 3], [true, 0n, 0]];
  assert(same(data.map(([n, c, q]) => Ieee754.text(new Ieee754.Datum(null, n, c, q))), ["1.5", "-1.50", "1E+2", "1.23E-7", "0E+3", "-0"]));
  assert(Ieee754.text(Ieee754.NAN) === "NaN" && same([...Ieee754.FORMATS.get("decimal64")!], [10n, 16, 384]));
  // A binary format's interchange encoding, big-endian, as an unsigned int: the canonical quiet NaN, subnormals, binary128.
  assert(Ieee754.to_bits("binary64", 1.5) === 4609434218613702656n && Ieee754.from_bits("binary64", 4609434218613702656n) === 1.5);
  assert(Ieee754.to_bits("binary64", -0) === 1n << 63n && Ieee754.to_bits("binary32", NaN) === 0x7FC00000n);
  assert(Ieee754.from_bits("binary32", 0x7F800000n) === Infinity && Number.isNaN(Ieee754.from_bits("binary32", 0xFF800001n)));
  assert(Ieee754.from_bits("binary16", 1n) === 2 ** -24 && Ieee754.to_bits("binary16", 65504.0) === 0x7BFFn);
  assert(Ieee754.to_bits("binary16", 2 ** -24) === 1n && Ieee754.to_bits("binary16", 0.0) === 0n);
  assert(Ieee754.to_bits("binary16", -Infinity) === 0xFC00n && Ieee754.WIDTHS.get("binary16") === 16);
  assert(Ieee754.to_bits("binary128", "1") === BigInt("0x3fff" + "00".repeat(14)));
  assert(Ieee754.from_bits("binary128", BigInt("0x3fff8" + "0".repeat(27))) === "1.5");
}

<a id="DOM-11"></a>
## DOM-11 · IEEE 754 arithmetic is exact, then rounded to the format by the domain's rounding direction

In [ ]:
{
  const [EVEN, AWAY, UP, DOWN, ZERO] = D.ROUNDINGS;
  const TABLE: [string, string, string, unknown, unknown, unknown][] = [ // format, rounding, operation, operands, result
    ["binary16", EVEN, "add", 1.0, 0.00048828125, 1.0],
    ["binary16", EVEN, "add", 1.0009765625, 0.00048828125, 1.001953125],
    ["binary16", AWAY, "add", 1.0, 0.00048828125, 1.0009765625],
    ["binary16", UP, "add", 1.0, 5.960464477539063e-08, 1.0009765625],
    ["binary16", DOWN, "add", 1.0, 5.960464477539063e-08, 1.0],
    ["binary16", ZERO, "add", 1.0, 5.960464477539063e-08, 1.0],
    ["binary16", DOWN, "add", -1.0, -5.960464477539063e-08, -1.0009765625],
    ["binary16", UP, "add", -1.0, -5.960464477539063e-08, -1.0],
    ["binary16", EVEN, "add", 65504.0, 65504.0, Infinity],
    ["binary16", ZERO, "add", 65504.0, 65504.0, 65504.0],
    ["binary16", UP, "add", 65504.0, 65504.0, Infinity],
    ["binary16", DOWN, "add", 65504.0, 65504.0, 65504.0],
    ["binary16", DOWN, "add", -65504.0, -65504.0, -Infinity],
    ["binary16", UP, "add", -65504.0, -65504.0, -65504.0],
    ["binary16", AWAY, "mul", 65504.0, 2.0, Infinity],
    ["binary16", EVEN, "mul", 6.103515625e-05, 0.0009765625, 5.960464477539063e-08],
    ["binary16", EVEN, "mul", 5.960464477539063e-08, 0.5, 0.0],
    ["binary16", AWAY, "mul", 5.960464477539063e-08, 0.5, 5.960464477539063e-08],
    ["binary16", EVEN, "add", 2047.0, 0.5, 2048.0],
    ["binary16", EVEN, "sub", 1.0, 1.0, 0.0],
    ["binary16", DOWN, "sub", 1.0, 1.0, -0],
    ["binary16", EVEN, "add", -0, -0, -0],
    ["binary16", EVEN, "add", 0.0, -0, 0.0],
    ["binary16", EVEN, "mul", -0, 5.0, -0],
    ["binary16", EVEN, "sub", Infinity, Infinity, NaN],
    ["binary16", EVEN, "add", Infinity, 1.0, Infinity],
    ["binary16", EVEN, "add", 1.0, -Infinity, -Infinity],
    ["binary16", EVEN, "add", NaN, 1.0, NaN],
    ["binary16", EVEN, "mul", Infinity, 0.0, NaN],
    ["binary16", EVEN, "mul", 0.0, -Infinity, NaN],
    ["binary16", EVEN, "mul", Infinity, -2.0, -Infinity],
    ["binary16", EVEN, "mul", NaN, 1.0, NaN],
    ["binary16", EVEN, "neg", 1.5, null, -1.5],
    ["binary16", EVEN, "neg", NaN, null, NaN],
    ["binary16", EVEN, "neg", -0, null, 0.0],
    ["binary32", EVEN, "add", 16777216.0, 1.0, 16777216.0],
    ["binary32", UP, "add", 16777216.0, 1.0, 16777218.0],
    ["binary32", EVEN, "mul", 0.10000000149011612, 3.0, 0.30000001192092896],
    ["binary64", ZERO, "add", 0.1, 0.2, 0.3],
    ["binary64", UP, "add", 0.1, 0.2, 0.30000000000000004],
    ["binary64", DOWN, "mul", 0.3333333333333333, 3.0, 0.9999999999999999],
    ["binary128", EVEN, "add", "0.1", "0.2", "0.30000000000000000000000000000000004"],
    ["binary128", EVEN, "mul", "1E+4932", "1E+1", "Infinity"],
    ["binary128", ZERO, "mul", "1E+4932", "1E+1", "1.189731495357231765085759326628007E+4932"],
    ["binary128", EVEN, "sub", "1", "1", "0"],
    ["binary128", DOWN, "sub", "1", "1", "-0"],
    ["binary128", EVEN, "neg", "-0", null, "0"],
    ["binary128", EVEN, "add", "Infinity", "1", "Infinity"],
    ["binary128", EVEN, "mul", "1E-4960", "1E-10", "0"],
    ["decimal64", EVEN, "add", "1.5", "1.50", "3.00"],
    ["decimal64", EVEN, "mul", "1.5", "1.50", "2.250"],
    ["decimal64", EVEN, "add", "9999999999999999", "1", "1.000000000000000E+16"],
    ["decimal64", EVEN, "add", "9999999999999999", "0.5", "1.000000000000000E+16"],
    ["decimal64", AWAY, "add", "1000000000000000", "0.5", "1000000000000001"],
    ["decimal64", EVEN, "add", "1000000000000000", "0.5", "1000000000000000"],
    ["decimal64", UP, "add", "1000000000000000", "0.1", "1000000000000001"],
    ["decimal64", DOWN, "add", "-1000000000000000", "-0.1", "-1000000000000001"],
    ["decimal64", ZERO, "add", "-1000000000000000", "-0.9", "-1000000000000000"],
    ["decimal64", EVEN, "mul", "1E+369", "1E+10", "1.0000000000E+379"],
    ["decimal64", EVEN, "add", "9.999999999999999E+384", "1E+369", "Infinity"],
    ["decimal64", ZERO, "add", "9.999999999999999E+384", "1E+369", "9.999999999999999E+384"],
    ["decimal64", EVEN, "mul", "1E-398", "0.1", "0E-398"],
    ["decimal64", UP, "mul", "1E-398", "0.1", "1E-398"],
    ["decimal64", EVEN, "mul", "1E-390", "0.00001", "1E-395"],
    ["decimal64", EVEN, "sub", "1.5", "1.5", "0.0"],
    ["decimal64", DOWN, "sub", "1.5", "1.5", "-0.0"],
    ["decimal64", EVEN, "add", "-0", "-0", "-0"],
    ["decimal64", EVEN, "mul", "-0.0", "1E+5", "-0E+4"],
    ["decimal64", EVEN, "mul", "1E+369", "0E+10", "0E+369"],
    ["decimal64", EVEN, "add", "NaN", "1", "NaN"],
    ["decimal64", EVEN, "mul", "Infinity", "0", "NaN"],
    ["decimal64", EVEN, "neg", "1.50", null, "-1.50"],
    ["decimal64", EVEN, "add", "0.000001", "1E-7", "0.0000011"],
    ["decimal64", EVEN, "mul", "123", "1E+5", "1.23E+7"],
    ["decimal128", EVEN, "mul", "0.1", "3", "0.3"],
    ["decimal128", EVEN, "add", "1E+6111", "1E+6111", "2E+6111"],
    ["decimal128", EVEN, "add", "1", "1E-40", "1.000000000000000000000000000000000"],
  ];
  /** The same value: floats by sign too, NaNs alike. */
  const alike = (x: unknown, y: unknown) => Object.is(x, y);
  // Python checks this table against its decimal module and the host's floats too (Python only).
  for (const [format, rounding, operation, a, b, expected] of TABLE) {
    const result = Ieee754.operate(operation, format, rounding, [a, b].filter((v) => v !== null));
    assert(alike(result, expected), `${format} ${a} ${b} ${result}`);
  }
  // Division, which Patterns has no operation for and C has (see C), by the same engine; decimals as divide does.
  const DIVISIONS: [string, unknown, unknown, unknown][] = [ // format, dividend, divisor, quotient, rounding ties to even
    ["binary64", NaN, 1.0, NaN], ["binary64", Infinity, Infinity, NaN], ["binary64", Infinity, -2.0, -Infinity], ["binary64", 1.0, Infinity, 0.0],
    ["binary64", -1.0, Infinity, -0], ["binary64", 1.0, 0.0, Infinity], ["binary64", -1.0, 0.0, -Infinity], ["binary64", 0.0, 0.0, NaN],
    ["binary64", 1.0, 3.0, 0.3333333333333333], ["binary32", 1.0, 3.0, 0.3333333432674408], ["decimal64", "1", "4", "0.25"],
    ["decimal64", "100", "10", "10"], ["decimal64", "1E+2", "1", "1E+2"], ["decimal64", "1", "3", "0.3333333333333333"],
    ["decimal64", "0", "5", "0"], ["decimal64", "0.00", "5", "0.00"], ["decimal64", "1", "Infinity", "0E-398"],
    ["decimal64", "Infinity", "2", "Infinity"], ["decimal64", "1", "0", "Infinity"], ["decimal64", "0", "0", "NaN"],
    ["decimal64", "-1", "8", "-0.125"], ["decimal64", "2.0", "4", "0.5"],
    ["decimal128", "1", "7", "0.1428571428571428571428571428571429"],
  ];
  for (const [format, a, b, expected] of DIVISIONS) assert(alike(Ieee754.operate("div", format, EVEN, [a, b]), expected), `${format} ${a} ${b}`);
  // Evaluation's floats keep the host's arithmetic, which binary64 rounding ties to even is.
  assert(ev("add", 0.1, 0.2) === 0.30000000000000004 && new D.OfIeee754.Data("binary64").equals(D.Float));
  // Conversions between formats and from and to integers, by the same engine (C's and SystemVerilog's casts use them).
  assert(Ieee754.convert("binary64", 0.1, "binary32", EVEN) === 0.10000000149011612);
  assert(Ieee754.convert("binary64", 0.1, "decimal64", EVEN) === "0.1000000000000000");
  assert(Ieee754.convert("binary64", 100.0, "decimal64", EVEN) === "100" && Ieee754.convert("binary64", 0.5, "decimal64", EVEN) === "0.5");
  assert(Ieee754.convert("decimal64", "1.50", "decimal128", EVEN) === "1.50" && Ieee754.convert("decimal64", "0.1", "binary64", EVEN) === 0.1);
  assert(Ieee754.convert("binary64", NaN, "binary128", EVEN) === "NaN" && Ieee754.convert("binary64", -Infinity, "decimal64", EVEN) === "-Infinity");
  assert(Ieee754.from_integer(2n ** 24n + 1n, "binary32", EVEN) === 16777216.0 && Ieee754.from_integer(2n ** 24n + 1n, "binary32", UP) === 16777218.0);
  assert(Ieee754.from_integer(10n ** 20n, "decimal64", EVEN) === "1.000000000000000E+20" && Ieee754.from_integer(-7n, "decimal64", EVEN) === "-7");
  assert(Ieee754.to_integer("binary64", 2.5, EVEN) === 2n && Ieee754.to_integer("binary64", -2.7, ZERO) === -2n);
  assert(Ieee754.to_integer("decimal64", "2.5", EVEN) === 2n && Ieee754.to_integer("binary64", Infinity, EVEN) === Infinity);
  assert(Ieee754.to_integer("binary64", -Infinity, EVEN) === -Infinity);
  raises(ValueError, () => Ieee754.to_integer("binary64", NaN, EVEN), "NaN has no integer value");
}

<a id="DOM-12"></a>
## DOM-12 · IEEE 754 values compare by totalOrder, cohorts by exponent, and NaNs equal each other

In [ ]:
{
  assert(D64.compare("1.5", "1.5") === 0 && D64.compare("1.5", "1.50") !== 0); // one value, two cohorts
  assert(D64.compare("1.50", "1.5") === -1 && D64.compare("-1.5", "-1.50") === -1);
  assert(D64.compare("1.5", "2") === -1 && D64.compare("-1", "-2") === 1);
  assert(D64.compare("-0", "0") === -1 && D64.compare("-Infinity", "-9.999999999999999E+384") === -1);
  assert(D64.compare("9.999999999999999E+384", "Infinity") === -1 && D64.compare("Infinity", "Infinity") === 0);
  assert(D64.compare("NaN", "NaN") === 0 && D64.compare("NaN", "1") === null && D64.compare("1", "NaN") === null);
  assert(D64.compare("1", "-1") === 1 && F128.compare("0.1", "0.2") === -1 && F128.compare("0.1", "0.1") === 0);
  assert(F16.compare(-0, 0) === -1 && F16.compare(NaN, NaN) === 0);
  // A binary128 value's text is the shortest that rounds back to it.
  const EVEN = "roundTiesToEven";
  assert(Ieee754.operate("mul", "binary128", EVEN, ["3", "0.5"]) === "1.5");
  assert(Ieee754.operate("add", "binary128", EVEN, ["0.1", "0.2"]) === "0.30000000000000000000000000000000004");
  assert(Ieee754.operate("mul", "binary128", EVEN, ["0.5", "0.5"]) === "0.25");
  assert(Ieee754.operate("neg", "binary128", EVEN, ["0.49"]) === "-0.49");
  assert(Ieee754.operate("mul", "binary128", EVEN, ["1E+48", "1E+1"]) === "1E+49"); // the nearest is below 1E+49
}


<a id="DOM-17"></a>
## DOM-17 · The standard dialects own the kinds and domains, with their inferred properties, read through the framework's interface

In [ ]:
{
  // Each standard dialect: a singleton of its own schema, a branch of the framework's kind union named by its tag, with
  // no terms; its standard by designation; its union of kinds the branch of the framework's named by its tag.
  const KINDS = new Map<any, [string, string, string[]]>([
    [ISOIEC11404, ["isoiec11404", "ISO/IEC 11404:2007", ["bool", "integer", "bits", "bytes", "enum", "packed"]]],
    [IEEE754, ["ieee754", "IEEE Std 754-2019", ["ieee754"]]], [IEEE1164, ["ieee1164", "IEEE Std 1164-1993", ["ieee1164"]]],
    [Unicode, ["unicode", "The Unicode Standard", ["unicode"]]]]);
  const Dialects = store.registered("Expressions.Framework.Dialect");
  for (const [module, [tag, standard, kinds]] of KINDS) {
    const path = `Expressions.${DIALECTS.get(module)}.Expressions`;
    assert(same(module.STANDARDS, [standard]) && await import(`@mbse/expressions/${DIALECTS.get(module)}/Expressions`) === module
      && (await import(`@mbse/expressions/${DIALECTS.get(module)}`)).Expressions === module); // its module, by its path and its family's
    const union = new Map<string, any>(DomainKind.branches.map((b: any) => [b.name, b.schema])).get(tag);
    assert(union === store.registered(`${path}.DomainKind`) && same(union.branches.map((b: any) => b.name), kinds));
    const dialect = new Map<string, any>(Dialects.branches.map((b: any) => [b.name, b.schema])).get(tag);
    assert(dialect === store.registered(`${path}.Dialect`) && dialect.ref && dialect.singleton === path);
    assert(same([...dialect.adjacencies.keys()], ["domains", "operations"])
      && FD.operations(singletons.get(DIALECTS.get(module) as string)).size === 0);
    raises(LookupError, () => store.registered(`${path}.Term`), `no schema registered as '${path}.Term'`);
  }
  // The kinds: their parameters, then the properties inferred from them, all optional.
  assert(same([...store.registered("Expressions.ISOIEC11404.Expressions.OfInteger").properties.keys()], [
    "width", "signed", "overflow", "least", "greatest", "ordered"]));
  assert(same([...store.registered("Expressions.IEEE754.Expressions.OfIeee754").properties.keys()], [
    "format", "rounding", "radix", "precision", "emax", "ordered"]));
  assert(equal([
    "Expressions.ISOIEC11404.Expressions.OfBool", "Expressions.ISOIEC11404.Expressions.OfBits",
    "Expressions.ISOIEC11404.Expressions.OfBytes", "Expressions.ISOIEC11404.Expressions.OfEnum",
    "Expressions.ISOIEC11404.Expressions.OfPacked", "Expressions.IEEE1164.Expressions.OfIeee1164",
    "Expressions.Unicode.Expressions.OfUnicode"].map((name) => [...store.registered(name).properties.keys()]), [
    ["ordered"], ["width", "ordered"], ["width", "ordered"], ["members", "ordered"], ["codes", "ordered"],
    ["width", "ordered"], ["ordered"]]));
  // Each domain's kind, through its dialect's union, and its properties, parameters and inferred ones alike, by name.
  assert(equal(new Map([...domains].map(([name, domain]) => [name, FD.kindOf(domain)[0]])), new Map([
    ["boolean", "bool"], ["integer", "integer"], ["octetstring", "bytes"], ["uint8", "integer"], ["binary64", "ieee754"],
    ["binary32", "ieee754"], ["std_logic", "ieee1164"], ["string", "unicode"]])));
  assert(FD.kindOf(domains.get("uint8"))[1].as_schema() === store.registered("Expressions.ISOIEC11404.Expressions.OfInteger"));
  const PROPERTIES = new Map<string, Map<string, unknown>>([
    ["boolean", new Map([["ordered", false]])], ["integer", new Map<string, unknown>([["signed", true], ["overflow", "raise"], ["ordered", true]])], // unbounded
    ["octetstring", new Map([["ordered", true]])],
    ["uint8", new Map<string, unknown>([["width", 8n], ["signed", false], ["overflow", "wrap"], ["least", 0n], ["greatest", 255n], ["ordered", true]])],
    ["binary64", new Map<string, unknown>([["format", "binary64"], ["rounding", "roundTiesToEven"], ["radix", 2n], ["precision", 53n], ["emax", 1023n], ["ordered", true]])],
    ["binary32", new Map<string, unknown>([["format", "binary32"], ["rounding", "roundTiesToEven"], ["radix", 2n], ["precision", 24n], ["emax", 127n], ["ordered", true]])],
    ["std_logic", new Map([["ordered", false]])], ["string", new Map([["ordered", true]])]]);
  assert(equal(new Map([...domains].map(([name, domain]) => [name, FD.properties(domain)])), PROPERTIES));
  assert(equal(FD.properties(state), new Map<string, unknown>([["members", ["IDLE", "RUN"]], ["ordered", false]]))); // a container as an array
  assert(equal(FD.properties(coded), new Map<string, unknown>([["codes", [0n, 1n]], ["ordered", true]])));
  // The inferred properties, populated by hand, agree with what the code domains compute.
  for (const [name, domain] of [...domains, ["state", state], ["coded", coded]] as [string, any][]) {
    assert(FD.properties(domain).get("ordered") === D.read(domain).kind().ORDERED, name);
  }
  const uint8 = PROPERTIES.get("uint8") as Map<string, unknown>;
  assert(equal([uint8.get("least"), uint8.get("greatest")], U8.bounds()) && equal(D.Int.bounds(), [null, null]));
  assert(["binary64", "binary32"].map((name) => PROPERTIES.get(name) as Map<string, unknown>).every((p) =>
    equal([p.get("radix"), p.get("precision"), p.get("emax")], Ieee754.FORMATS.get(p.get("format") as string))));
  // A domain's name, its dialect's `Defines` entry's; null until a dialect defines it.
  assert(same([...domains.values()].map((domain) => FD.nameOf(domain)), [...domains.keys()]) && FD.nameOf(coded) === "coded");
  const loose = store.transaction(() => {
    const loose = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union(
      (u: any) => u.unicode((v: any) => v.unicode((e: any) => e))))));
    assert(FD.nameOf(loose) === null);
    store.Expressions.Unicode.Expressions.Dialect(singletons.get("Unicode"), (b: any) => b.domains({ loose: (e: any) => e.domain(loose) }));
    return loose;
  });
  assert(FD.nameOf(loose) === "loose" && FD.properties(loose).size === 0); // no property set
  raises(ValueError, () => FD.kindOf(empty), "a domain needs a kind");
  raises(ValueError, () => FD.properties(empty), "a domain needs a kind");
  // A dialect's operations by name, in order, each its expression schema, and each one's arity: Patterns'.
  const patterns = store.transaction(Patterns.define)[0].get("dialect");
  const operations = FD.operations(patterns);
  assert(same([...operations.keys()], [...E.CORE.keys()]) && operations.get("add") === store.registered("Expressions.Framework.Patterns.add"));
  assert(equal(new Map([...operations].map(([name, operation]) => [name, FD.arity(operation)])), E.CORE));
  // Patterns owns no domain, and holds no kind and no domain's schema.
  assert([...patterns.domains].length === 0 && DomainKind.branches[0].schema.branches.length > 0
    && !DomainKind.branches.map((b: any) => b.name).includes("patterns"));
  for (const name of ["Expressions.Framework.Patterns.uint8", "Expressions.Framework.Patterns.OfInteger", "Expressions.Framework.Patterns.Packs"]) {
    raises(LookupError, () => store.registered(name), `no schema registered as '${name}'`);
  }
}


<a id="DOM-18"></a>
## DOM-18 · A language's own kinds are its dialect's: SystemVerilog's logic, Python3's and TypeScript5's strings, the tokens and Excel's errors

In [ ]:
{
  // Each dialect with kinds of its own adds its union of them to the framework's, as the branch named by its tag: a
  // language's own types (Decided, B and H), each kind's parameters, then its inferred properties, all optional.
  const languages: any = new Builders.Store();
  for (const install of [...INSTALLS, Python3.install, Matlab.install, Excel.install, Latex.install, CDialect.install, Cpp.install,
    SystemVerilog.install, TypeScript5.install]) {
    languages.transaction(install);
  }
  const LANGUAGES = new Map<string, [string, Map<string, [string, string[]]>]>([ // each dialect's tag, its path, and its kinds' properties, by tag
    ["python3", ["Expressions.Python.Python3", new Map([["str", ["OfStr", ["ordered"]]], ["nonetype", ["OfNoneType", ["ordered"]]],
      ["ellipsis", ["OfEllipsis", ["ordered"]]]])]],
    ["excel", ["Expressions.Excel.Expressions", new Map([["error", ["OfError", ["members", "ordered"]]]])]],
    ["c", ["Expressions.C.Expressions", new Map([["nullptr_t", ["OfNullptr", ["ordered"]]]])]],
    ["cpp", ["Expressions.Cpp.Expressions", new Map([["nullptr_t", ["OfNullptr", ["ordered"]]]])]],
    ["systemverilog", ["Expressions.SystemVerilog.Expressions", new Map([
      ["logic", ["OfLogic", ["width", "signed", "least", "greatest", "ordered"]]]])]],
    ["typescript5", ["Expressions.TypeScript.TypeScript5", new Map([["string", ["OfString", ["ordered"]]], ["null", ["OfNull", ["ordered"]]]])]],
  ]);
  const Kinds = languages.registered("Expressions.Framework.DomainKind");
  assert(same(Kinds.branches.map((b: any) => b.name), ["isoiec11404", "ieee754", "ieee1164", "unicode", ...LANGUAGES.keys()])); // MATLAB, LaTeX: none
  for (const [tag, [path, kinds]] of LANGUAGES) {
    const union = new Map<string, any>(Kinds.branches.map((b: any) => [b.name, b.schema])).get(tag);
    assert(union === languages.registered(`${path}.DomainKind`));
    assert(equal(new Map(union.branches.map((b: any) => [b.name, [b.schema.name, [...b.schema.properties.keys()]]])),
      new Map([...kinds].map(([kind, [schema, names]]) => [kind, [`${path}.${schema}`, names]]))));
  }
  const members = languages.registered("Expressions.Excel.Expressions.OfError").properties.get("members").schema;
  assert(members.index === languages.registered("Schemas.OfInt") && members.element === languages.registered("Schemas.OfStr"));
  // C's nullptr_t and C++'s are each their own dialect's (API, 35).
  assert(languages.registered("Expressions.C.Expressions.OfNullptr") !== languages.registered("Expressions.Cpp.Expressions.OfNullptr"));
  // A domain of each, written by hand in a transaction and owned by its dialect by name, valid against its schema;
  // `ordered` true of the strings and SystemVerilog's logic, false of the tokens and Excel's errors.
  const ERRORS = ["#DIV/0!", "#N/A", "#NAME?", "#NULL!", "#NUM!", "#REF!", "#VALUE!"];
  const WRITTEN: [[string, string], [string, (e: any) => any, Map<string, unknown>]][] = [ // by its dialect's tag and its name: its kind, its properties written, and as read
    [["systemverilog", "logic8"], ["logic", (e: any) => e.width(8n).signed(false).least(0n).greatest(255n).ordered(true),
      new Map<string, unknown>([["width", 8n], ["signed", false], ["least", 0n], ["greatest", 255n], ["ordered", true]])]],
    [["python3", "str"], ["str", (e: any) => e.ordered(true), new Map([["ordered", true]])]],
    [["python3", "NoneType"], ["nonetype", (e: any) => e.ordered(false), new Map([["ordered", false]])]],
    [["python3", "ellipsis"], ["ellipsis", (e: any) => e.ordered(false), new Map([["ordered", false]])]],
    [["typescript5", "string"], ["string", (e: any) => e.ordered(true), new Map([["ordered", true]])]],
    [["typescript5", "null"], ["null", (e: any) => e.ordered(false), new Map([["ordered", false]])]],
    [["c", "nullptr_t"], ["nullptr_t", (e: any) => e.ordered(false), new Map([["ordered", false]])]],
    [["cpp", "nullptr_t"], ["nullptr_t", (e: any) => e.ordered(false), new Map([["ordered", false]])]],
    [["excel", "error"], ["error", (e: any) => e.members(ERRORS).ordered(false), new Map<string, unknown>([["members", ERRORS], ["ordered", false]])]],
  ];

  /** The builder of a schema registered under a dotted path, by its parts. */
  const schemaAt = (path: string): any => path.split(".").reduce((found: any, part) => found[part], languages);

  const LanguageDomain = languages.registered("Expressions.Framework.Domain");
  const written: any[] = languages.transaction(() => {
    const domains = WRITTEN.map(([[tag], [kind, spec]]) => languages.Expressions.Framework.Domain((d: any) => d.kind(
      (k: any) => k.as_value((a: any) => a.as_union((u: any) => u[tag]((v: any) => v[kind](spec)))))));
    WRITTEN.forEach(([[tag, name]], i) => { // each owned: written with its owner
      const path = (LANGUAGES.get(tag) as [string, unknown])[0];
      schemaAt(`${path}.Dialect`)(languages.singleton(path), (b: any) => b.domains({ [name]: (e: any) => e.domain(domains[i]) }));
    });
    return domains;
  });
  WRITTEN.forEach(([[tag, name], [kind, , read]], i) => {
    const domain = written[i];
    assert(Validators.Validate(languages)(LanguageDomain, domain).length === 0, `${tag} ${name}`);
    const [found, record] = FD.kindOf(domain);
    const [path, kinds] = LANGUAGES.get(tag) as [string, Map<string, [string, string[]]>];
    assert(found === kind && record.as_schema() === languages.registered(`${path}.${(kinds.get(kind) as [string, string[]])[0]}`));
    assert(equal(FD.properties(domain), read) && FD.nameOf(domain) === name);
  });
}
